# EDA preliminar — Colegios EC

Análisis exploratorio inicial de `data/processed/Preliminar colegios EC.xlsx`, fuente interina mientras se habilita la tabla en SQL Server (ver `etl/build_resumen.py`, cuya conexión no se toca aquí). Los nombres de columna de este archivo son los mismos que tendrá esa tabla.

**Grano:** estudiante × carrera (`IdBanner` + `CodCarreraBanner_Sales`). 17,084 filas × 30 columnas.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "app"))

from utils.data_loader import PRELIMINAR_COLEGIOS_PATH  # noqa: E402
from utils.theme import COLOR_PRIMARIO, COLOR_SECUNDARIO  # noqa: E402

pd.set_option("display.max_columns", None)

COLOR_TERCIARIO = "#5D9BD5"  # mismo tono que el embudo de Home.py
COLOR_ALERTA = "#B0745A"     # terracota apagado, para el estado "Retiro"

df = pd.read_excel(PRELIMINAR_COLEGIOS_PATH)
print(f"Filas: {len(df):,} — Columnas: {df.shape[1]}")
df.head()

2026-07-15 15:56:55.059 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-15 15:56:55.061 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-15 15:56:55.062 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Filas: 17,084 — Columnas: 30


,PeriodoBanner_Sales,AñoGraduacionAMIED,ZonaInecAMIED,RegimenAMIED,ProvinciaAMIED,CantonAMIED,Sostenimiento,CodColegioBanner_Sales,CodColegioAMIED,NombreInstitucionAMIED,Cluster,BACHILLERATO PENSIÓN,RangoPension,EstudiantesFemeninoTercerAñoBACH,EstudiantesMasculinoTercerAñoBACH,GraduadosColegioAMIED,TotalDocumentadosPeriodo,IdBanner,AdEtapaReal,AdEtapaInicio,CodCarreraBanner_Sales,Colegiatura,ColegiaturaMenosBeca,AdIndRetiro,AdEstado,FechaInicioAcademico,MINEDUFechaGrado,DiferenciaMeses,Diferencia,CedulaPasaporte
0,202310,2022.0,Urbana,Sierra,PICHINCHA,QUITO,Particular,17H00220,17H00220,UNIDAD EDUCATIVA PARTICULAR SANTA MARÍA EUFRASIA,NaN,193.79,NaN,87.0,46.0,133.0,37,A00106522,6,6,UDLA1P734,4100.00,3362.00,0,PagoCompleto,2022-09-26,NaN,NaN,ES MENOS DE 1 AÑO,1723351316
1,202310,2022.0,Urbana,Sierra,PICHINCHA,QUITO,Particular,17H00220,17H00220,UNIDAD EDUCATIVA PARTICULAR SANTA MARÍA EUFRASIA,NaN,193.79,NaN,87.0,46.0,133.0,37,A00444584,6,6,UDLA1H564,3209.00,3209.00,0,PagoCompleto,2022-09-26,NaN,NaN,ES MENOS DE 1 AÑO,1752755114
2,202310,2022.0,Urbana,Sierra,PICHINCHA,QUITO,Particular,17H00220,17H00220,UNIDAD EDUCATIVA PARTICULAR SANTA MARÍA EUFRASIA,NaN,193.79,NaN,87.0,46.0,133.0,37,A00095366,6,6,UDLA1P734,4100.00,3280.00,0,PagoCompleto,2022-09-26,NaN,NaN,ES MENOS DE 1 AÑO,1725650640
3,202310,2022.0,Urbana,Sierra,PICHINCHA,QUITO,Particular,17H00220,17H00220,UNIDAD EDUCATIVA PARTICULAR SANTA MARÍA EUFRASIA,NaN,193.79,NaN,87.0,46.0,133.0,37,A00108366,6,6,UDLA1P791,3092.78,2845.36,0,PagoCompleto,2022-09-26,NaN,NaN,ES MENOS DE 1 AÑO,1723007652
4,202320,2022.0,Urbana,Sierra,PICHINCHA,QUITO,Fiscal,17H00232,17H00232,UNIDAD EDUCATIVA LICEO POLICIAL,NaN,90.00,NaN,77.0,110.0,187.0,7,A00113332,6,6,UDLA1P052,3765.48,3012.38,0,PagoCompleto,2023-03-23,NaN,NaN,ES MENOS DE 1 AÑO,1755201942


## 1. Resumen general

Tipos de dato y proporción de nulos por columna.

In [2]:
resumen_cols = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "n_unicos": df.nunique(),
})
resumen_cols.sort_values("pct_nulos", ascending=False)

,dtype,pct_nulos,n_unicos
Cluster,float64,100.0,0
RangoPension,float64,100.0,0
DiferenciaMeses,float64,86.9,138
MINEDUFechaGrado,object,86.9,262
ProvinciaAMIED,object,66.1,24
RegimenAMIED,object,66.1,2
EstudiantesFemeninoTercerAñoBACH,float64,66.1,222
CantonAMIED,object,66.1,122
Sostenimiento,object,66.1,4
CodColegioAMIED,object,66.1,815


## 2. Cobertura de colegio vinculado

Los campos `*AMIED` (provincia, cantón, sostenimiento, etc.) solo se llenan cuando el registro pudo vincularse a un colegio de origen. **Esto limita cualquier corte geográfico al subconjunto vinculado.**

In [3]:
total = len(df)
con_colegio = df["CodColegioAMIED"].notna().sum()
sin_colegio = total - con_colegio

cobertura = pd.DataFrame({
    "grupo": ["Con colegio vinculado", "Sin colegio vinculado"],
    "n": [con_colegio, sin_colegio],
})
cobertura["pct"] = (cobertura["n"] / total * 100).round(1)
print(cobertura.to_string(index=False))

fig_cobertura = go.Figure(
    go.Bar(
        x=cobertura["n"],
        y=cobertura["grupo"],
        orientation="h",
        marker_color=[COLOR_PRIMARIO, COLOR_SECUNDARIO],
        text=cobertura.apply(lambda r: f"{r['n']:,} ({r['pct']}%)", axis=1),
        textposition="outside",
    )
)
fig_cobertura.update_layout(
    template="plotly_white",
    title="Cobertura de vínculo a colegio de origen",
    xaxis_title="Estudiantes",
    yaxis_title=None,
    showlegend=False,
    margin=dict(l=10, r=80, t=60, b=10),
)
fig_cobertura.show()

                grupo     n  pct
Con colegio vinculado  5784 33.9
Sin colegio vinculado 11300 66.1


## 3. Estado del proceso (`AdEstado`)

Distribución de estudiantes por estado de pago/retiro, sobre el total de filas (no depende de tener colegio vinculado).

In [4]:
orden_estado = ["PagoCompleto", "Regularización", "PosibleRetiro", "Retiro"]
color_estado = {
    "PagoCompleto": COLOR_PRIMARIO,
    "Regularización": COLOR_TERCIARIO,
    "PosibleRetiro": COLOR_SECUNDARIO,
    "Retiro": COLOR_ALERTA,
}

estado_counts = df["AdEstado"].value_counts(dropna=False).reindex(orden_estado + [pd.NA]).dropna(how="all")
estado_df = df["AdEstado"].fillna("Sin dato").value_counts().reindex(orden_estado + ["Sin dato"]).dropna().reset_index()
estado_df.columns = ["AdEstado", "n"]
estado_df["pct"] = (estado_df["n"] / total * 100).round(1)

fig_estado = go.Figure(
    go.Bar(
        x=estado_df["AdEstado"],
        y=estado_df["n"],
        marker_color=[color_estado.get(e, COLOR_SECUNDARIO) for e in estado_df["AdEstado"]],
        text=estado_df.apply(lambda r: f"{r['n']:,} ({r['pct']}%)", axis=1),
        textposition="outside",
    )
)
fig_estado.update_layout(
    template="plotly_white",
    title="Distribución por estado del proceso",
    xaxis_title=None,
    yaxis_title="Estudiantes",
    showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_estado.show()
estado_df

,AdEstado,n,pct
0,PagoCompleto,16012,93.7
1,Regularización,30,0.2
2,PosibleRetiro,494,2.9
3,Retiro,541,3.2
4,Sin dato,7,0.0


## 4. Financiero: beca aplicada

`beca_aplicada = Colegiatura - ColegiaturaMenosBeca`. Se excluyen las 326 filas sin dato de colegiatura.

In [5]:
fin = df.dropna(subset=["Colegiatura", "ColegiaturaMenosBeca"]).copy()
fin["beca_aplicada"] = fin["Colegiatura"] - fin["ColegiaturaMenosBeca"]
fin["beca_pct"] = (fin["beca_aplicada"] / fin["Colegiatura"]).replace([float("inf"), -float("inf")], pd.NA)

print(f"Estudiantes con dato financiero: {len(fin):,}")
print(f"Con alguna beca aplicada (>0): {(fin['beca_aplicada'] > 0).sum():,} ({(fin['beca_aplicada'] > 0).mean():.1%})")
display(fin[["Colegiatura", "ColegiaturaMenosBeca", "beca_aplicada", "beca_pct"]].describe())

fig_beca = go.Figure(
    go.Histogram(
        x=fin.loc[fin["beca_aplicada"] > 0, "beca_pct"] * 100,
        marker_color=COLOR_PRIMARIO,
        nbinsx=30,
    )
)
fig_beca.update_layout(
    template="plotly_white",
    title="Distribución del % de beca aplicada (solo estudiantes con beca > 0)",
    xaxis_title="% de beca sobre colegiatura",
    yaxis_title="Estudiantes",
    showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_beca.show()

Estudiantes con dato financiero: 16,758
Con alguna beca aplicada (>0): 13,026 (77.7%)


,Colegiatura,ColegiaturaMenosBeca,beca_aplicada,beca_pct
count,16758.000000,16758.000000,16758.000000,16162.000000
mean,3397.278249,2646.639737,750.638512,0.231797
std,940.149893,1062.500968,569.669398,0.165881
min,0.000000,0.000000,0.000000,0.000000
25%,3121.780000,2069.022500,344.950000,0.149999
50%,3430.500000,2527.170000,759.540000,0.220000
75%,3716.000000,3121.780000,1114.880000,0.350000
max,5301.790000,4912.340000,3888.990000,0.999997


## 5. Tendencia por periodo (`PeriodoBanner_Sales`)

Volumen de estudiantes y tasa de `PagoCompleto` por periodo. Dos gráficos de un solo eje cada uno (evitar doble eje).

In [6]:
por_periodo = (
    df.assign(es_pago_completo=df["AdEstado"] == "PagoCompleto")
    .groupby("PeriodoBanner_Sales")
    .agg(n_estudiantes=("IdBanner", "count"), tasa_pago_completo=("es_pago_completo", "mean"))
    .reset_index()
    .sort_values("PeriodoBanner_Sales")
)

fig_vol = go.Figure(
    go.Scatter(
        x=por_periodo["PeriodoBanner_Sales"].astype(str),
        y=por_periodo["n_estudiantes"],
        mode="lines+markers",
        line=dict(color=COLOR_PRIMARIO, width=2),
        marker=dict(size=8, color=COLOR_PRIMARIO, line=dict(width=2, color="white")),
    )
)
fig_vol.update_layout(
    template="plotly_white",
    title="Estudiantes por periodo",
    xaxis_title="Periodo",
    yaxis_title="Estudiantes",
    showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_vol.show()

fig_tasa = go.Figure(
    go.Scatter(
        x=por_periodo["PeriodoBanner_Sales"].astype(str),
        y=por_periodo["tasa_pago_completo"],
        mode="lines+markers",
        line=dict(color=COLOR_TERCIARIO, width=2),
        marker=dict(size=8, color=COLOR_TERCIARIO, line=dict(width=2, color="white")),
    )
)
fig_tasa.update_layout(
    template="plotly_white",
    title="Tasa de PagoCompleto por periodo",
    xaxis_title="Periodo",
    yaxis_title="Tasa",
    yaxis_tickformat=".0%",
    showlegend=False,
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_tasa.show()

## 6. Antigüedad bachillerato → matrícula (`Diferencia`)

¿El tiempo transcurrido entre graduación de bachillerato y matrícula se relaciona con el estado del proceso?

In [7]:
cruce = pd.crosstab(df["Diferencia"], df["AdEstado"], normalize="index") * 100
cruce = cruce.reindex(columns=[c for c in orden_estado if c in cruce.columns])
cruce.round(1)

AdEstado,PagoCompleto,Regularización,PosibleRetiro,Retiro
Diferencia,,,,
ES MENOS DE 1 AÑO,93.6,0.2,3.0,3.2
MAS DE 1 AÑO,95.5,0.1,1.2,3.2


## 7. Corte geográfico (solo subconjunto con colegio vinculado)

**Cuidado:** cubre solo el ~33.9% de las filas (ver sección 2).

In [8]:
con_colegio_df = df[df["CodColegioAMIED"].notna()].copy()
con_colegio_df["es_pago_completo"] = con_colegio_df["AdEstado"] == "PagoCompleto"

por_provincia = (
    con_colegio_df.groupby("ProvinciaAMIED")
    .agg(n_estudiantes=("IdBanner", "count"), tasa_pago_completo=("es_pago_completo", "mean"))
    .sort_values("n_estudiantes", ascending=False)
    .head(10)
    .reset_index()
)

fig_prov = go.Figure(
    go.Bar(
        x=por_provincia["n_estudiantes"],
        y=por_provincia["ProvinciaAMIED"],
        orientation="h",
        marker_color=COLOR_PRIMARIO,
        text=por_provincia["n_estudiantes"],
        textposition="outside",
    )
)
fig_prov.update_layout(
    template="plotly_white",
    title="Top 10 provincias por estudiantes (con colegio vinculado)",
    xaxis_title="Estudiantes",
    yaxis_title=None,
    yaxis=dict(autorange="reversed"),
    showlegend=False,
    margin=dict(l=10, r=60, t=60, b=10),
)
fig_prov.show()
por_provincia

,ProvinciaAMIED,n_estudiantes,tasa_pago_completo
0,PICHINCHA,4633,0.962875
1,IMBABURA,175,0.908571
2,TUNGURAHUA,110,0.945455
3,COTOPAXI,102,0.980392
4,CHIMBORAZO,102,0.950980
5,SANTO DOMINGO DE LOS TSACHILAS,94,0.968085
6,MANABI,82,0.963415
7,CARCHI,70,0.971429
8,LOJA,58,0.982759
9,GUAYAS,54,0.870370


In [9]:
por_sostenimiento = (
    con_colegio_df.groupby("Sostenimiento")
    .agg(n_estudiantes=("IdBanner", "count"), tasa_pago_completo=("es_pago_completo", "mean"))
    .sort_values("n_estudiantes", ascending=False)
    .reset_index()
)
por_sostenimiento["tasa_pago_completo"] = (por_sostenimiento["tasa_pago_completo"] * 100).round(1)
por_sostenimiento

,Sostenimiento,n_estudiantes,tasa_pago_completo
0,Particular,3824,96.3
1,Fiscal,1214,94.8
2,Fiscomisional,544,96.1
3,Municipal,202,97.5


## 8. Limitaciones y próximos pasos

- **66.1% de las filas no tienen colegio vinculado** — el corte geográfico/sostenimiento solo aplica al 33.9% restante; cualquier lectura sobre provincia/cantón debe leerse con esa salvedad.
- `Cluster` y `RangoPension` vienen 100% vacías — reservadas para llenarse más adelante, no son utilizables todavía.
- El campo `BACHILLERATO PENSIÓN` de este archivo no coincide en nombre con `PENSIÓN`, que es el que espera `COLUMNAS_COLEGIO` en `etl/build_resumen.py` — pendiente confirmar si es el mismo dato antes de unificar.
- Este archivo no trae `Base`, `Afluente`, `Matriculado`, `Documentado` (las columnas que arma el embudo de `Home.py` vía `construir_resumen()`), por lo que **no reemplaza** el pipeline SQL actual — es un análisis paralelo a nivel estudiante mientras se habilita la tabla real en SQL Server.
- Siguiente paso sugerido: una vez resuelto el mapeo de `PENSIÓN`/`BACHILLERATO PENSIÓN` y la cobertura de colegio, decidir si este archivo alimenta una página nueva en la app o si se espera a la tabla SQL definitiva.